# Baseline — Predict Blindness Before It Happens

**Competition:** grade retina photographs for **diabetic retinopathy** severity on the
APTOS 2019 scale: 0 (none) to 4 (proliferative). Early detection prevents blindness.

- **Task:** 5-class ordinal classification (2000 train / 500 test images)
- **Metric:** quadratic weighted kappa (QWK) — being off by 1 grade is much better
  than being off by 3
- **Kaggle link:** _TODO: add link_

**Approach:** frozen ImageNet ResNet-18 features + Logistic Regression. Note the heavy
class imbalance (grade 0 dominates).

In [1]:
import numpy as np
import pandas as pd
import torch
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image

DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(train.shape, test.shape, train["diagnosis"].value_counts().to_dict())

(2000, 2) (500, 1) {0: 986, 2: 546, 1: 202, 4: 161, 3: 105}


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
weights = ResNet18_Weights.IMAGENET1K_V1
backbone = resnet18(weights=weights)
backbone.fc = torch.nn.Identity()
backbone.eval().to(device)
preprocess = weights.transforms()

@torch.no_grad()
def extract_features(ids, batch_size=32):
    feats = []
    for i in range(0, len(ids), batch_size):
        batch = [preprocess(Image.open(f"{DATA_DIR}/images/{x}.jpg").convert("RGB"))
                 for x in ids[i:i+batch_size]]
        feats.append(backbone(torch.stack(batch).to(device)).cpu().numpy())
    return np.vstack(feats)

X  = extract_features(train["id"].tolist())
Xt = extract_features(test["id"].tolist())
print(X.shape, Xt.shape)

(2000, 512) (500, 512)


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import cohen_kappa_score, accuracy_score

y = train["diagnosis"].values
clf = LogisticRegression(max_iter=3000, class_weight="balanced")
oof = cross_val_predict(clf, X, y, cv=5)
print(f"CV accuracy: {accuracy_score(y, oof):.4f}")
print(f"CV quadratic weighted kappa: {cohen_kappa_score(y, oof, weights='quadratic'):.4f}")

CV accuracy: 0.7455
CV quadratic weighted kappa: 0.8157


In [4]:
clf.fit(X, y)
sub = pd.DataFrame({"id": test["id"], "diagnosis": clf.predict(Xt)})
sub.to_csv("submission.csv", index=False)
sub.head()

,id,diagnosis
0,6cbc3dad809c,2
1,1006345f70b7,2
2,75a4343b12f9,1
3,2974c6ad1d58,0
4,abdb365cacbc,2


## Ideas to improve

- Fine-tune the CNN on GPU with augmentation (the original APTOS winners used
  EfficientNet ensembles).
- Treat the problem as **regression + thresholding** — QWK rewards ordinal-aware
  predictions; optimize the thresholds on out-of-fold predictions.
- Preprocess retina images: crop the circular fundus, Ben Graham color normalization.
